In [1]:
try:
    %load_ext autoreload --quiet
except ImportError:
    %reload_ext autoreload
%autoreload 2

import hashlib
from typing import List, Any
from flow_merge.lib.snapshot.data_architecture._normalized_slices import NormalizedSlice
from pydantic import BaseModel, computed_field, Field
import datetime
from pathlib import Path
import json

class MergePlan(BaseModel):
    created_at: datetime.datetime = Field(default=datetime.datetime.now())
    base_model: str
    tokenizer_mode: str
    tokenizer_interpolation_method: str
    slices: List[NormalizedSlice]
    lib_version: str

    @classmethod
    def from_config(cls, config: Any) -> "MergePlan":
        return cls(
            created_at=datetime.datetime.now(),
            base_model=config.base_model,
            tokenizer_mode=config.tokenizer_mode,
            tokenizer_interpolation_method=config.tokenizer_interpolation_method,
            slices=config.slices,
            lib_version=config.lib_version,
        )

    @classmethod
    def from_file(cls, file_path: Path | str) -> "MergePlan":
        with open(Path(file_path).resolve(), "rb") as f:
            parsed = json.load(f)
            return cls(**parsed)

    @computed_field
    @property
    def sha(self) -> str:
        obj = self.model_dump_json(exclude={"created_at", "sha"}).encode("utf-8")
        return hashlib.md5(obj).hexdigest()

mp = MergePlan.from_file("../flow_merge/lib/example-slices.json")


In [2]:
print(mp.sha)

dacbc64dcbfa795dc7eb9a2c267bf051


In [10]:
from flow_merge.lib.snapshot.data_architecture._normalized_slices import NormalizedSource
from torch import Tensor
from flow_merge.lib.loaders.normalizer import Source, MergeMethod
from flow_merge.lib.model.architecture import ModelWeightArch
from flow_merge.lib.tensor.loader import TensorRepository
from toolz import compose
from flow_merge.lib.merge_methods import method_classes, method_configs, MergeMethodIdentifier
from typing import Tuple, Dict, Optional
from flow_merge.lib.model import Model
from flow_merge.lib.tokenizer import get_merge_tokenizer, Tokenizer

class StagedSource(BaseModel, arbitrary_types_allowed=True):
    layer: Optional[str]
    range: Optional[List[int]]
    model: str
    is_base: Optional[bool]
    weight: float
    tensor: Tensor

class StagedSlice(BaseModel):
    merge_method: MergeMethod
    sources: List[StagedSource]
    base_layer_name: Optional[str] # the name of the layer which is used as base

    def merge(self) -> Tensor:
        return self.merge_method.method.merge(self)
    
    
class Stage(BaseModel):
    plan: MergePlan
    models: List[Dict[str, Model]]
    tokenizer: Tokenizer
    slices: List[StagedSlice]
    
    @classmethod
    def from_plan(cls, plan) -> "Stage":
        models: List[Dict[str, Model]] = cls._models_from_slices(plan.slices)
        tokenizer = cls._create_common_tokenizer(plan.slices, models, plan.tokenizer_mode)
        slices: List[StagedSlice] = [cls.transform(x) for x in plan.slices]
        return cls(plan, models, tokenizer, slices)
    
    def run(self) -> None:
        for staged_slice in self.slices:
            staged_slice.merge()
    
    # materialize models and tokenizer    
    @staticmethod
    def _models_from_slices(slices):
        all_model_path_occurrences = [model_field.model for x in slices for model_field in x.sources]
        all_distinct_model_paths = list({model for model in all_model_path_occurrences})
        return [{model_path: Model.from_path(model_path)} for model_path in all_distinct_model_paths]
    
    @staticmethod 
    def _create_common_tokenizer(slices, models, tokenizer_mode):
        all_models = [(model_field.model, model_field.is_base) for x in slices for model_field in x.sources]
        the_most_frequent_base_model_name = max(set(name for name, is_base in all_models if is_base),
                                        key=lambda name: sum(is_base for n, is_base in all_models if n == name),
                                        default=None)
        base_model = next((model for model in models.values() if model.id == the_most_frequent_base_model_name), None)
        return get_merge_tokenizer(models=models.values(), base_model=base_model, tokenizer_mode=tokenizer_mode)
    
    # stage the slices
    def transform(self) -> StagedSlice:
        return compose(
            self._add_tensors, 
            self._add_merge_method
        )

    @staticmethod
    def _add_tensors(slice_obj):
        for src in slice_obj.sources:
            src.tensor = TensorRepository.get_tensor(
                    shards=src.model.shards,
                    tensor_key=src.model.architecture.get_weight(src.layer).name,
                )
        return slice_obj

    @staticmethod
    def _add_merge_method(self, slice_obj):
        method_settings = method_configs[slice_obj.merge_method.name]
        slice_obj.merge_method: MergeMethod = MergeMethod(name=slice_obj.merge_method.name,
                                             method=method_classes[slice_obj.merge_method.name],
                                             settings=method_settings(**slice_obj.merge_method.params))
        return slice_obj
    
    # TODO!
    
    # MISSING STATIC METHOD?
    # def _get_merged_config(self):
    # merged_model_config = base_model.architecture.config
    # merged_model_config.num_hidden_layers = num_hidden_layers

    # EXCEPTION CASE - INTERPOLATION
    # def create_executable_interpolation_slice(self, slice_obj):
    #     if slice_obj.merge_method.name == MergeMethodIdentifier.INTERPOLATE:
    #         slice_obj.merge_method = {"name": slice_obj.merge_method.name,
    #                                   "method": None,
    #                                   "settings": None}
    # 
    #         if self.tokenizer.input_ids_mappings:
    #             merged_model_config.vocab_size = len(
    #                 self.tokenizer.tokenizer.get_vocab()
    #             )
    # 
    #         all_tensors = \
    #             { base_model: base_model_tensor, **models_tensors }
    #         hidden_dim = _validate_tensor_shapes(
    #             base_model_weight=, # ModelWeight
    #             tensors=all_tensors, # Dict[Model, torch, Tensor]
    #             base_model_layer_type=task_base_model_weight.layer_type # str
    #         )
    #     return slice_obj

    # TAKE THIS ELSEWHERE / REWRITE
    # @staticmethod
    # def _validate_tensor_shapes(
    #         base_model_weight: ModelWeight,
    #         tensors: Dict[Model, torch.Tensor],
    #         base_model_layer_type: str
    # ):
    #     hidden_size = next(iter(tensors.values())).shape[
    #         1 if base_model_layer_type == "embedding" else 0
    #     ]
    #     for model, tensor in tensors.items():
    #         current_size = tensor.shape[1] if base_model_layer_type == "embedding" else tensor.shape[0]
    #         if current_size != hidden_size:
    #             raise RuntimeError(
    #                 f"Tensor shape mismatch in '{base_model_weight.name}'. Expected {hidden_size}, but {model.path} has {current_size}."
    #             )
    #     return hidden_size


    # def interpolate(
    #         cls,
    #         base_model: Model,
    #         all_tensors: Dict[Model, torch.Tensor],
    #         method_config,
    #         input_ids_mappings: Dict[Model, Dict[int, int]],
    #         sources: List[NormalizedSource],
    #         hidden_dim: int
    # ):
    # InterpolationRunner.interpolate(
    #     base_model=base_model,
    #     all_tensors=all_tensors,
    #     merge_method=method_config,
    #     input_ids_mappings=tokenizer.input_ids_mappings,
    #     sources=sources,
    #     hidden_dim=hidden_dim
    # )



        
